<a href="https://colab.research.google.com/github/NeevWadhwa-Helloworld/Machine-Learning-Projects/blob/main/Language_Translation_using_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Language Translation with Deep Learning**

In [ ]:
import numpy as np
import pandas as pd
import os
import string
from string import digits
import matplotlib.pyplot as plt
%matplotlib inline
import re
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from keras.layers import Input,LSTM,Embedding,Dense
from keras.models import Model

In [ ]:
lines=pd.read_csv("/content/Requirments (4).zip",encoding='utf-8')
print(lines.head())

      source                                   english_sentence  \
0        ted  politicians do not have permission to do what ...   
1        ted         I'd like to tell you about one such child,   
2  indic2012  This percentage is even greater than the perce...   
3        ted  what we really mean is that they're bad at not...   
4  indic2012  .The ending portion of these Vedas is called U...   

                                      hindi_sentence  
0  राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह कर...  
1  मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहू...  
2   यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।  
3     हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते  
4        इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।  


In [ ]:
lines=lines[lines["source"]=='ted']
lines=lines[~pd.isnull(lines['english_sentence'])]
lines.drop_duplicates(inplace=True)
lines=lines.sample(n=25000,random_state=42)
lines.shape

(25000, 3)

In [ ]:
lines["english_sentence"]=lines["english_sentence"].apply(lambda x: x.lower())
lines["hindi_sentence"]=lines["hindi_sentence"].apply(lambda x:x.lower())

In [ ]:
import re
# Use \. to escape the period, or [.] to match a literal dot
lines['english_sentence'] = lines['english_sentence'].apply(lambda x: re.sub(r'\.', '', x))
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: re.sub(r'\.', '', x))

In [ ]:
exclude = set(string.punctuation)
lines['english_sentence']=lines["english_sentence"].apply(lambda x:''.join(ch for ch in x if ch not in exclude))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x:''.join(ch for ch in x if ch not in exclude))

In [ ]:
remove_digits=str.maketrans('','',digits)
lines['english_sentence']=lines['english_sentence'].apply(lambda x:x.translate(remove_digits))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x:x.translate(remove_digits))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x:re.sub("[२३०८१५७९४६]",'',x))
lines['english_sentence']=lines['english_sentence'].apply(lambda x:x.strip())
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x:x.strip())
lines['english_sentence']=lines['english_sentence'].apply(lambda x:re.sub(" +"," ",x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x:re.sub(" +"," ",x))
lines["hindi_sendtence"]=lines["hindi_sentence"].apply(lambda x:'START_'+x+'_END')


In [ ]:
all_eng_words = set()
for eng in lines['english_sentence']:
    for word in eng.split():
        all_eng_words.add(word)

all_hindi_words = set()
for hin in lines['hindi_sendtence']:
    for word in hin.split():
        all_hindi_words.add(word)

lines['lines_eng_sentence'] = lines['english_sentence'].apply(lambda x: len(x.split()))
lines['lines_hin_sentence'] = lines['hindi_sendtence'].apply(lambda x: len(x.split()))

In [ ]:
lines=lines[lines['lines_eng_sentence']<=20]
lines=lines[lines['lines_hin_sentence']<=20]
max_length_src=max(lines['lines_hin_sentence'])
max_length_tar=max(lines['lines_eng_sentence'])
input_words=sorted(list(all_eng_words))
target_words=sorted(list(all_hindi_words))
num_encoder_tokens=len(all_eng_words)
num_decoder_tokens=len(all_hindi_words)
num_encoder_tokens,num_decoder_tokens

num_decoder_tokens+=1
input_token_index=dict([(word,i+1)for i,word in enumerate(input_words)])
target_token_index=dict([(word,i+1) for i, word in enumerate(target_words)])
reverse_input_char_index=dict((i, word) for word, i  in input_token_index.items())
reverse_target_char_index=dict((i, word) for word, i in target_token_index.items())
lines=shuffle(lines)

## **Training Model to Translate English to Hinde**

In [ ]:
x, y = lines['english_sentence'], lines['hindi_sendtence']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
X_train.to_pickle('X_train.pkl')
X_test.to_pickle('X_test.pkl')

In [ ]:
def generate_batch(X_data, y_data, batch_size=128):
    while True:
        for j in range(0, len(X_data), batch_size):
            # Determine actual batch size for the last chunk
            current_batch_size = min(batch_size, len(X_data) - j)

            encoder_input_data = np.zeros((current_batch_size, max_length_tar), dtype='float32')
            decoder_input_data = np.zeros((current_batch_size, max_length_src), dtype='float32')
            decoder_target_data = np.zeros((current_batch_size, max_length_src, num_decoder_tokens), dtype='float32')

            X_chunk = X_data[j:j+current_batch_size]
            y_chunk = y_data[j:j+current_batch_size]

            for i, (input_text, target_text) in enumerate(zip(X_chunk, y_chunk)):
                for t, word in enumerate(input_text.split()):
                    if t < max_length_tar:
                        encoder_input_data[i, t] = input_token_index.get(word, 0)

                for t, word in enumerate(target_text.split()):
                    if t < max_length_src:
                        token_idx = target_token_index.get(word, 0)
                        decoder_input_data[i, t] = token_idx
                        if t > 0:
                            decoder_target_data[i, t - 1, token_idx] = 1
            # Yield a tuple (inputs, targets) where inputs is also a tuple
            yield ((encoder_input_data, decoder_input_data), decoder_target_data)

latent_dim = 300
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens + 1, latent_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(num_decoder_tokens, latent_dim, mask_zero=True)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')
train_samples = len(X_train)
val_samples = len(X_test)
batch_size = 128
epochs = 100

In [ ]:
model.fit(generate_batch(X_train, y_train, batch_size=batch_size),
          steps_per_epoch=train_samples // batch_size,
          epochs=epochs,
          validation_data=generate_batch(X_test, y_test, batch_size=batch_size),
          validation_steps=val_samples // batch_size)
model.save_weights('nmt_weights.h5')

Epoch 1/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 1965s 13s/step - loss: 6.9156 - val_loss: 6.3161
Epoch 2/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 1883s 12s/step - loss: 6.2750 - val_loss: 6.2879
Epoch 3/100
143/155 ━━━━━━━━━━━━━━━━━━━━ 2:09 11s/step - loss: 6.2421

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2)
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_token_index['START_']
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += ' ' + sampled_char
        if (sampled_char == '_END' or len(decoded_sentence) >50):
          stop_condition = True
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]
    return decoded_sentence
    train_gen=generate_batch(X_train,y_train,batch_size=1)
k=-1

In [ ]:
k+=1
(input_seq,actual_output),_=next(train_gen)
decoded_sentence=decode_sequence(input_seq)
print("Input English sentence:",X_train[k:k+1].values[0])
print("Actual Hindi Translation:",y_train[k:k+1].values[0][6:-4])
print("Predicted Hindi Translation:",decoded_sentence[:-4])